# A free λ sweep from one SMC run — FLUX.1-dev

Comparing λ values used to cost one full `adaptive_tempering_smc_sample` run each, and the runs were not
strictly comparable: each had its own β schedule, its own resampling history, its own ancestry.

Adaptive tempering already walks β from 0 to 1, and level *k* holds a particle cloud targeting

$$\pi_{\beta_{k}} \propto p(x) \exp(\beta_{k} \lambda_{\max} f(x)) = q_{\lambda_{\text{eff}}},
\qquad \lambda_{\text{eff}} = \beta_{k} \cdot \lambda_{\max}$$

So a single run at $\lambda_{\max}$ **already contains the whole sweep** — it was thrown away because only the
final cloud was returned. `keep_levels=True` captures each level, turning one run into a λ sweep at exactly
the same reward, references and base density.

**This notebook:** run FLUX.1-dev once at $\lambda_{\max}$, decode every level through the VAE, and render a
$\lambda_{\text{eff}} \times$ particle grid.

**Two caveats, by construction (not bugs):**

- **Rows share ancestry.** A row is *not* an independent draw from $q_{\lambda_{\text{eff}}}$, and consecutive
  rows are strongly correlated. We order grid columns by lineage so that correlation is legible instead of hidden.
- **λ_eff values are chosen by the ESS rule**, not by us. A finer `ess_target` yields more rows.

In [ ]:
import math
import os
import sys
import time

T_START = time.time()

# This notebook lives in creativity-measure/notebooks/flux_lambda_sweep/, so the repo root is two
# levels up. The package is installed editable (`pip install -e '.[flux]'`) so the import works
# without this; kept as a fallback for a fresh checkout.
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isdir(os.path.join(REPO, "creativity_measure")):
    REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))  # if cwd is already notebooks/
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Latents, checkpoints and rendered output all live next to this notebook.
SWEEP_DIR = os.path.join(REPO, "notebooks", "flux_lambda_sweep")
os.makedirs(SWEEP_DIR, exist_ok=True)

# --- wall-clock budget and the internal deadline -------------------------------------------------
# The SMC is one long library call, so it cannot be interrupted from outside. Instead the
# checkpointing kernel checks DEADLINE after every rejuvenation sweep and stops cleanly, leaving
# enough time to decode and render whatever levels finished. RESERVE_H is that headroom.
#
# Under Slurm, SLURM_JOB_END_TIME is the job's real end (epoch seconds), so the deadline tracks
# --time automatically and cannot drift out of sync with it.
WALL_BUDGET_H = 7.0     # keep in sync with --time in flux_lambda_sweep.slurm (420 min)
RESERVE_H = 0.75        # held back for VAE decoding, rendering, and writing the notebook

try:
    _slurm_end = float(os.environ["SLURM_JOB_END_TIME"])
except (KeyError, ValueError):
    _slurm_end = None
if _slurm_end and _slurm_end > T_START:
    DEADLINE = _slurm_end - RESERVE_H * 3600
    WALL_BUDGET_H = (_slurm_end - T_START) / 3600      # the true remaining budget
    _deadline_src = "SLURM_JOB_END_TIME"
else:
    DEADLINE = T_START + (WALL_BUDGET_H - RESERVE_H) * 3600
    _deadline_src = "WALL_BUDGET_H constant"

import torch

torch.set_num_threads(os.cpu_count() or 1)
torch.manual_seed(0)

from creativity_measure import set_default_dtype

# FLUX runs in bf16, but everything the SMC touches is fp32 (see the dtype trap in the adapter cell).
set_default_dtype(torch.float32)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.bfloat16  # transformer/VAE compute dtype

# The token lives in the private $HOME (chmod 600), NOT under $WORK, which is group-readable.
# ~/.cshrc sets HF_TOKEN_PATH to it; HF_HOME still points at $WORK so the 32G of weights land there.
print("repo          =", REPO)
print("sweep dir     =", SWEEP_DIR)
print("HF_HOME       =", os.environ.get("HF_HOME"))
print("HF_TOKEN_PATH =", os.environ.get("HF_TOKEN_PATH"))
print(f"budget        = {WALL_BUDGET_H:.2f} h  (reserve {RESERVE_H:.2f} h)  via {_deadline_src}")
print(f"SMC deadline  = {time.strftime('%H:%M:%S', time.localtime(DEADLINE))}")

from huggingface_hub import auth_check

# Preflight: a gated-access failure here costs 2 seconds. The same failure after the pipeline
# starts loading costs minutes -- and on a preemptible partition, possibly the whole allocation.
auth_check("black-forest-labs/FLUX.1-dev")
print("FLUX.1-dev access: OK")
print("device:", DEVICE, "| compute dtype:", DTYPE)

## 1. Load FLUX.1-dev and encode the prompt once

The prompt is frozen for the whole run, so both text encoders are needed exactly once. Encoding first and
then **deleting** them frees ~9.5 GB (T5 alone is ~8.9 GB in bf16), which is what makes a 48 GB card
comfortable alongside the ~24 GB transformer.

In [ ]:
from diffusers import FluxPipeline

MODEL_ID = "black-forest-labs/FLUX.1-dev"
PROMPT = "A dog"
GUIDANCE = 3.5  # FLUX.1-dev is guidance-distilled: the scale is an *embedding*, not a CFG pass

t0 = time.time()
print(f"Loading {MODEL_ID} in {DTYPE} ...")
pipe = FluxPipeline.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
pipe.to(DEVICE)
print(f"  loaded in {time.time() - t0:.0f}s")

with torch.no_grad():
    prompt_embeds, pooled_prompt_embeds, text_ids = pipe.encode_prompt(
        prompt=PROMPT, prompt_2=None, device=DEVICE, max_sequence_length=512
    )
print(f"prompt_embeds {tuple(prompt_embeds.shape)}  (T5)")
print(f"pooled        {tuple(pooled_prompt_embeds.shape)}  (CLIP)")
print(f"text_ids      {tuple(text_ids.shape)}")

# Free both text encoders -- the embeddings above are all we need from them.
before = torch.cuda.memory_allocated() / 2**30 if DEVICE.type == "cuda" else 0.0
pipe.text_encoder = None
pipe.text_encoder_2 = None
pipe.tokenizer = None
pipe.tokenizer_2 = None
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    after = torch.cuda.memory_allocated() / 2**30
    print(f"text encoders freed: {before:.1f} GB -> {after:.1f} GB allocated")

transformer = pipe.transformer.eval()
vae = pipe.vae.eval()
print("transformer guidance_embeds =", transformer.config.guidance_embeds)

## 2. Adapter: FLUX flow-matching velocity → EDM denoiser $D(x_{\sigma}, \sigma) = E[X \mid x_{\sigma}]$

This is the one genuinely risky part. The repo's machinery (`edm_generator`, `edm_score_fn`) speaks the EDM
convention; FLUX.1-dev speaks rectified flow matching. The map between them is exact.

**Noise conventions.** Flow matching interpolates, EDM adds:

$$x_{t}^{\text{FM}} = (1 - t) x_{0} + t \varepsilon,
\qquad x_{\sigma}^{\text{EDM}} = x_{0} + \sigma \varepsilon$$

Factoring out $(1-t)$ gives $x_{t} = (1-t)(x_{0} + \frac{t}{1-t}\varepsilon)$, so the two agree under

$$\sigma = \frac{t}{1 - t}, \qquad t = \frac{\sigma}{1 + \sigma},
\qquad x_{t} = \frac{x_{\sigma}}{1 + \sigma}$$

**Velocity → denoiser.** FLUX predicts $v = \varepsilon - x_{0}$, so

$$x_{t} - t v = (1-t)x_{0} + t\varepsilon - t\varepsilon + t x_{0} = x_{0}$$

i.e. $\hat{x}_{0} = x_{t} - t v$ is exactly the MMSE estimate the EDM denoiser must return. No approximation.

**Why $\sigma_{\max} = 80$ is the natural choice here.** At $\sigma = 80$, $t = 80/81 \approx 0.988$ and
$x_{t} = 80 z / 81 \approx z$ — unit-scale noise, precisely what FLUX expects at $t \to 1$. Unlike the
Stable-Diffusion path there is no schedule-derived $\sigma_{\max}$ to look up; the VAE's `scaling_factor`
already normalizes latents to roughly unit variance.

**Calls the transformer needs** (verified against the installed diffusers 0.39.0 source):

- `hidden_states`: latents **packed** into a patch sequence, $(B, 16, 64, 64) \to (B, 1024, 64)$
- `timestep`: $t$ in $[0, 1]$ — `forward` multiplies by 1000 internally, so do **not** pre-scale
- `guidance`: likewise scaled by 1000 internally; pass 3.5, not 3500
- `img_ids` / `txt_ids`: **2-D** (no batch dim) in 0.39.0 — a 3-D tensor triggers a deprecation warning

**Dtype trap.** The transformer runs in bf16 but the denoiser returns **fp32**. `reward.x_refs` pins device
*and* dtype for the entire SMC run, and ESS/softmax/weight arithmetic in bf16 would quietly corrupt the β
schedule — bf16 has ~3 decimal digits of mantissa, and the tempering step solves for a target ESS.

In [ ]:
from creativity_measure.distances.edm_adapter import edm_score_fn
from creativity_measure.generators.base import edm_generator

# 512x512: VAE downsamples 8x -> 64x64 latent, 16 channels. d = 65536.
# (1024x1024 would be d = 262144, and pCN mixes worse in higher dimension.)
IMG = 512
C, H, W = 16, IMG // 8, IMG // 8
LATENT_DIM = C * H * W
SIGMA_MIN, SIGMA_MAX = 2e-3, 80.0
N_STEPS = 8  # Heun ODE steps inside G

# Positional ids. Both must be 2-D in diffusers 0.39.0.
# _prepare_latent_image_ids takes the *packed* grid, i.e. half the latent resolution.
img_ids = FluxPipeline._prepare_latent_image_ids(1, H // 2, W // 2, DEVICE, DTYPE)
txt_ids = text_ids if text_ids.ndim == 2 else text_ids[0]
print(f"img_ids {tuple(img_ids.shape)}  txt_ids {tuple(txt_ids.shape)}  "
      f"(packed seq len = {(H // 2) * (W // 2)})")


def denoiser(x: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
    """EDM denoiser D(x_sigma, sigma) = E[X | x_sigma] backed by the FLUX transformer.

    x:     (B, 16, 64, 64) fp32, EDM convention  x_sigma = x_0 + sigma * eps
    sigma: (B,)
    returns fp32, same shape as x.
    """
    b = x.shape[0]
    sig = sigma.reshape(b, 1, 1, 1).to(torch.float32)

    # EDM -> flow matching.  t = sigma/(1+sigma),  x_t = x_sigma/(1+sigma).
    t = (sigma / (1.0 + sigma)).to(torch.float32)          # (B,) in [0, 1)
    x_t = x.to(torch.float32) / (1.0 + sig)

    packed = FluxPipeline._pack_latents(x_t.to(DTYPE), b, C, H, W)
    guidance = torch.full((b,), GUIDANCE, device=x.device, dtype=torch.float32)

    with torch.no_grad():
        v = transformer(
            hidden_states=packed,
            timestep=t.to(DTYPE),                          # forward() multiplies by 1000 itself
            guidance=guidance,                             # ditto -- pass 3.5, not 3500
            pooled_projections=pooled_prompt_embeds.expand(b, -1).to(DTYPE),
            encoder_hidden_states=prompt_embeds.expand(b, -1, -1).to(DTYPE),
            txt_ids=txt_ids,
            img_ids=img_ids,
            return_dict=False,
        )[0]

    v = FluxPipeline._unpack_latents(v, IMG, IMG, 8).to(torch.float32)   # (B, 16, 64, 64)

    # velocity -> denoised sample:  x_0 = x_t - t * v   (exact, see the cell above)
    x0 = x_t - t.reshape(b, 1, 1, 1) * v
    return x0.to(torch.float32)                            # fp32 out: pins the SMC's dtype


# G(z): (B, d) -> (B, d) in FLUX latent space, and the Tweedie score for the IEM geometry.
G = edm_generator(
    denoiser, img_shape=(C, H, W), sigma_min=SIGMA_MIN, sigma_max=SIGMA_MAX, n_steps=N_STEPS
)
score_fn = edm_score_fn(denoiser, (C, H, W))
print(f"generator + score ready   d = {LATENT_DIM}, sigma in [{SIGMA_MIN}, {SIGMA_MAX}], "
      f"n_steps = {N_STEPS}")

## 3. Go / no-go gate — **run this before submitting any long job**

`G(randn(4))` → VAE decode → display. If these are not clean, prompt-compliant images, the velocity →
denoiser mapping is wrong and **nothing downstream means anything**: the reward would be measuring IEM
geometry of a broken score field, and the whole sweep would be noise dressed up as a result.

Also times a single `G` call, which is the unit the whole budget is built from.

In [ ]:
import matplotlib.pyplot as plt

VAE_SF = vae.config.scaling_factor      # 0.3611 for FLUX
VAE_SHIFT = vae.config.shift_factor     # 0.1159 -- SD has no shift; FLUX does
print(f"vae scaling_factor={VAE_SF}  shift_factor={VAE_SHIFT}")


def decode(latents_flat: torch.Tensor, chunk: int = 2) -> torch.Tensor:
    """Flat FLUX latents (B, 65536) -> pixel images (B, 3, 512, 512) in [0, 1].

    Chunked because the VAE decoder is the memory spike, not the transformer.
    """
    outs = []
    for i in range(0, latents_flat.shape[0], chunk):
        lat = latents_flat[i : i + chunk].reshape(-1, C, H, W).to(DEVICE, DTYPE)
        lat = lat / VAE_SF + VAE_SHIFT          # undo the normalization the model works in
        with torch.no_grad():
            img = vae.decode(lat).sample
        outs.append(pipe.image_processor.postprocess(img.float(), output_type="pt").cpu())
    return torch.cat(outs)


def show(imgs: torch.Tensor, titles=None, ncol: int = 4, scale: float = 2.6) -> None:
    n = imgs.shape[0]
    nrow = math.ceil(n / ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * scale, nrow * scale))
    for k, ax in enumerate(getattr(axes, "flat", [axes])):
        ax.axis("off")
        if k < n:
            ax.imshow(imgs[k].permute(1, 2, 0).clamp(0, 1).numpy())
            if titles is not None:
                ax.set_title(titles[k], fontsize=9)
    plt.tight_layout()
    plt.show()


# --- the gate ---------------------------------------------------------------------------------
gen = torch.Generator(device="cpu").manual_seed(7)
z0 = torch.randn(4, LATENT_DIM, generator=gen).to(DEVICE)

t0 = time.time()
with torch.no_grad():
    x0 = G(z0)
g_secs = time.time() - t0
print(f"G(randn(4)) took {g_secs:.1f}s  ->  {g_secs / 4:.1f}s per sample, "
      f"{g_secs / (4 * N_STEPS):.2f}s per transformer call")
print(f"latent stats: mean={x0.mean():.3f}  std={x0.std():.3f}   "
      f"(std should be ~1 if the sigma<->t mapping is right)")

show(decode(x0.cpu()), titles=[f"G sample {i}" for i in range(4)])

## 4. Reward, γ window, and the λ scale

`SquaredGlobalIEMDistance` + `NormalizedExpectedDistanceReward` — the current default per the project's
CLAUDE.md. The reward is unitless, so λ acts as a plain inverse temperature.

The γ window starts from the **data** length scale $S = \operatorname{std}(x_{\text{refs}})$ rather than
$\sigma_{\max}$ (which is pure noise, $\gg S$), and is clamped inside
$[1/\sigma_{\max}^{2}, 1/\sigma_{\min}^{2}]$ — the denoiser's valid range, as `edm_score_fn` requires.

**Choosing $\lambda_{\max}$.** The scale unit is $\lambda_{s} = 1/\operatorname{std}_{p}(f)$, and the
bracketing rule is $m \cdot \lambda_{s}$ for $m \in \{1, 2, 3\}$, so $\lambda_{\max} = 3 \lambda_{s}$.
The ESS probe below is the check on that: if `ESS/N` at $\lambda_{\max}$ is near 1 the tilt does not bite,
adaptive tempering terminates in a single level, and **there is no sweep to harvest** — the cell warns
explicitly if that happens.

In [ ]:
from creativity_measure import SquaredGlobalIEMDistance
from creativity_measure.tilt import NormalizedExpectedDistanceReward

# =================================================================================================
# COST KNOBS -- every expensive decision in this notebook lives here.
# (WALL_BUDGET_H / RESERVE_H / DEADLINE come from the environment cell, so the budget has a single
#  source of truth and tracks Slurm's --time automatically.)
#
# One reward call on a batch of B costs, in single-image transformer passes:
#     R * (N_GAMMA - 1) * NUM_EPS * (1 + B)
# because per reference, per gamma-interval, iem_sq_increments_one_to_many makes two score calls:
# batch NUM_EPS for the reference and batch NUM_EPS*B for the particles.
#
# One G(z) call on a batch of B costs (2*N_STEPS - 1) * B passes -- Heun does 2 denoiser evals per
# step, minus the corrector skipped at sigma=0. Each pCN sweep pays BOTH a G call and a reward call.
#
# The normalizer E_pair[D^2] is an R x R term paid ONCE, inside
# NormalizedExpectedDistanceReward.__post_init__:  R * (N_GAMMA - 1) * NUM_EPS * (1 + R)
#
# The NUMBER OF LEVELS is not knowable in advance -- the ESS rule decides it while walking beta from
# 0 to 1 -- so cost is expressed PER LEVEL and inverted into "how many levels does the budget buy".
# =================================================================================================
R = 4            # references. Cheapest thing to cut: normalizer noise only RESCALES f, and the
                 # lam_s = 1/std(f) calibration absorbs that rescaling automatically.
N_GAMMA = 6      # gamma-integration nodes. Too few is a SYSTEMATIC quadrature bias in the metric
                 # (left-endpoint rule over N_GAMMA-1 intervals), and bias does not average out.
NUM_EPS = 3      # Brownian paths. Also sets the inner score batch NUM_EPS*N (= 24 here, not 128).

N = 8            # SMC particles: 8 grid columns, and uniq/N granularity
MAX_SWEEPS = 16  # cap on adaptive rejuvenation -- the thing NOT to economise on

MIN_USEFUL_LEVELS = 6      # fewer lambda_eff rows than this is not a sweep worth running
ABORT_IF_OVER_BUDGET = True

# NOTE on fidelity: the Brownian bank is SEEDED (seed=123 below), so small NUM_EPS / N_GAMMA do not
# make f stochastic -- f stays deterministic, as the SMC theory requires. They make it a coarser
# approximation of the true IEM distance: we sample correctly from a slightly different target,
# rather than incorrectly from the right one. That is why cutting these is safer than cutting sweeps.


def _reward_passes(batch: int, n_calls: int = 1) -> int:
    """Single-image transformer passes for `n_calls` reward calls on `batch` particles."""
    return R * (N_GAMMA - 1) * NUM_EPS * (1 + batch) * n_calls


def _g_passes(batch: int, n_calls: int = 1) -> int:
    """Passes for `n_calls` G(z) calls on `batch` latents (Heun: 2 evals/step, minus final corrector)."""
    return (2 * N_STEPS - 1) * batch * n_calls


NORM_PASSES = _reward_passes(R)                          # R x R normalizer, once
REF_PASSES = _g_passes(R)                                # generating the references
SWEEP_PASSES = _reward_passes(N) + _g_passes(N)          # one pCN sweep: G call + reward call
LEVEL_PASSES = SWEEP_PASSES * MAX_SWEEPS                 # one level, worst case (all sweeps at cap)
# Paid before any level: normalizer, references, the lambda probe below, and the SMC's kernel.init.
FIXED_PASSES = NORM_PASSES + REF_PASSES + 2 * SWEEP_PASSES

# Budget available to the SMC itself, i.e. after the reserve held back for decode/render/write.
SMC_BUDGET_H = WALL_BUDGET_H - RESERVE_H


def affordable_levels(sec_per_pass: float, budget_h: float | None = None) -> float:
    """How many tempering levels the budget buys at this measured speed."""
    b = SMC_BUDGET_H if budget_h is None else budget_h
    return (b * 3600 / sec_per_pass - FIXED_PASSES) / LEVEL_PASSES


# --- PREFLIGHT: time one score call, project, and bail out BEFORE paying the normalizer ----------
# This must come first: the R x R normalizer is charged inside the reward constructor below, so a
# projection printed after that point would arrive too late to be useful.
_gamma_probe = torch.tensor(1.0, device=DEVICE)
_y = torch.randn(NUM_EPS, LATENT_DIM, device=DEVICE)
if DEVICE.type == "cuda":
    score_fn(_y, _gamma_probe)          # warm-up: kernel autotuning, allocator growth
    torch.cuda.synchronize()
t0 = time.time()
score_fn(_y, _gamma_probe)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
t_per_pass = (time.time() - t0) / NUM_EPS     # seconds per single-image pass

n_fit = affordable_levels(t_per_pass)
print(f"PREFLIGHT   {t_per_pass * 1000:.0f} ms per single-image pass "
      f"(batch {NUM_EPS} score call, warmed up)")
print(f"  knobs: R={R} N_GAMMA={N_GAMMA} NUM_EPS={NUM_EPS} N={N} "
      f"cap={MAX_SWEEPS} n_steps={N_STEPS}")
print(f"  budget {WALL_BUDGET_H:.2f} h - reserve {RESERVE_H:.2f} h = {SMC_BUDGET_H:.2f} h for SMC")
print(f"  fixed (normalizer + refs + probe + init) : {FIXED_PASSES:>10,} passes  "
      f"{FIXED_PASSES * t_per_pass / 3600:6.2f} h")
print(f"  per sweep  (G + reward)                  : {SWEEP_PASSES:>10,} passes  "
      f"{SWEEP_PASSES * t_per_pass / 60:6.2f} min")
print(f"  per level  (at cap {MAX_SWEEPS:2d})                  : {LEVEL_PASSES:>10,} passes  "
      f"{LEVEL_PASSES * t_per_pass / 3600:6.2f} h")
print(f"  -> buys {n_fit:.1f} levels (need >= {MIN_USEFUL_LEVELS})")
print(f"  (at 2x this speed: {affordable_levels(t_per_pass * 2):.1f} levels)")
if n_fit < MIN_USEFUL_LEVELS:
    msg = (f"budget buys only {n_fit:.1f} tempering levels, below MIN_USEFUL_LEVELS="
           f"{MIN_USEFUL_LEVELS}: too few lambda_eff rows to be a sweep. Cut NUM_EPS first (it also "
           f"shrinks the NUM_EPS*N={NUM_EPS * N} score batch, so it buys OOM headroom too), then "
           f"N_GAMMA, then R. Do NOT lower MAX_SWEEPS -- adaptive_rejuvenation.ipynb shows too few "
           f"sweeps under-tilt badly. Alternatively raise --time, or use --partition=gpu-b200 "
           f"(5-day limit, not preemptible, faster card).")
    if ABORT_IF_OVER_BUDGET:
        raise RuntimeError(msg)
    print("WARNING:", msg)
print()

# --- references, gamma window, reward ------------------------------------------------------------
with torch.no_grad():
    latent_refs = G(torch.randn(R, LATENT_DIM, generator=gen).to(DEVICE))

S = latent_refs.std().item()                      # per-coordinate data length scale
GAMMA_LO = max(1.0 / S**2, 1.0 / SIGMA_MAX**2)
GAMMA_HI = min(2.0**10, 1.0 / SIGMA_MIN**2)
GAMMAS = torch.logspace(math.log2(GAMMA_LO), math.log2(GAMMA_HI), N_GAMMA, base=2).to(DEVICE)

D2 = SquaredGlobalIEMDistance(None, GAMMAS, num_eps=NUM_EPS, seed=123, score_fn=score_fn)

t0 = time.time()
reward = NormalizedExpectedDistanceReward(distance=D2, x_refs=latent_refs)   # pays the R x R term
t_norm = time.time() - t0

print(f"refs {tuple(latent_refs.shape)}  data scale S={S:.3f}  "
      f"gamma window=[{GAMMA_LO:.3g}, {GAMMA_HI:.3g}] x {N_GAMMA} nodes")
print(f"reward dtype/device pinned by x_refs: {latent_refs.dtype} / {latent_refs.device}")
print(f"normalizer took {t_norm:.0f} s  (preflight predicted {NORM_PASSES * t_per_pass:.0f} s)")

# --- lambda scale + ESS probe --------------------------------------------------------------------
# N_PROBE == N, so this timing IS the per-sweep cost of the run (one G call + one reward call).
# Caveat: std(f) from 8 samples has ~27% relative error (1/sqrt(2(n-1))), and that propagates
# straight into LAM_MAX -- treat it as order-of-magnitude, not precise.
N_PROBE = N
t0 = time.time()
with torch.no_grad():
    f_probe = reward(G(torch.randn(N_PROBE, LATENT_DIM, generator=gen).to(DEVICE)))
t_sweep = time.time() - t0
print(f"one sweep-equivalent (G + reward) on batch {N_PROBE} took {t_sweep:.0f} s "
      f"(preflight predicted {SWEEP_PASSES * t_per_pass:.0f} s)")

f_std = float(f_probe.std())
lam_s = (1.0 / f_std) if f_std > 0 else float("inf")
print(f"\nf ~ p:  mean={float(f_probe.mean()):.4f}  std={f_std:.4f}  "
      f"range=[{float(f_probe.min()):.4f}, {float(f_probe.max()):.4f}]")
print(f"lambda_s = 1/std(f) = {lam_s:.1f}   (+-27% from 8 samples)\n")

# ESS/N here is the one-shot importance-sampling ESS from p straight to q_lambda -- exactly what the
# first tempering step faces. Reuses f_probe, so it costs no extra model calls.
print(f"{'m':>4} {'lambda':>10} {'ESS/N':>7}")
for m in (1.0, 2.0, 3.0, 10.0, 30.0, 100.0):
    w = torch.softmax(m * lam_s * f_probe, dim=0)
    print(f"{m:4.0f} {m * lam_s:10.1f} {float(1.0 / (w.pow(2).sum() * N_PROBE)):7.2f}")

LAM_MAX = 3.0 * lam_s
ess_at_max = float(1.0 / (torch.softmax(LAM_MAX * f_probe, dim=0).pow(2).sum() * N_PROBE))
print(f"\nLAM_MAX = 3 * lambda_s = {LAM_MAX:.1f}   (ESS/N at that lambda: {ess_at_max:.2f})")
if ess_at_max > 0.9:
    print("WARNING: tilt barely bites -> expect very few levels, i.e. almost no sweep to harvest.")
    print("         Raise m (see the table) before committing to the long run.")

# --- refined projection, from the measured per-sweep time ----------------------------------------
t_level = t_sweep * MAX_SWEEPS
fit_measured = (SMC_BUDGET_H * 3600 - t_norm - 2 * t_sweep) / t_level
print(f"\n{'=' * 74}\nMEASURED PROJECTION   SMC budget = {SMC_BUDGET_H:.2f} h")
print(f"  normalizer (already paid) : {t_norm / 60:6.1f} min")
print(f"  per sweep                 : {t_sweep / 60:6.2f} min")
print(f"  per level (at cap {MAX_SWEEPS:2d})     : {t_level / 60:6.1f} min")
print(f"  -> buys {fit_measured:.1f} levels; the kernel stops cleanly at the deadline either way\n")
print(f"{'levels':>7} {'sweeps':>8} {'projected':>12}  {'verdict':>8}")
for lv in (4, 6, 8, 10, 12, 16):
    hours = (t_norm + 2 * t_sweep + lv * t_level) / 3600
    print(f"{lv:7d} {lv * MAX_SWEEPS:8d} {hours:10.1f} h  "
          f"{'OK' if hours <= SMC_BUDGET_H else 'OVER':>8}")
print(f"{'=' * 74}")

## 5. The run — one call, `keep_levels=True`, checkpointed as it goes

`ess_target=0.85` is deliberately finer than usual: a tighter ESS floor means smaller β steps, hence **more
levels, hence more $\lambda_{\text{eff}}$ rows** in the sweep. That is the whole point here — the extra levels
are what we are harvesting, not overhead.

**`n_mcmc=None` — adaptive rejuvenation length, capped at 16.** Each level sweeps until pCN's own
geometry-aware `decorrelation` metric falls below `decorr_threshold` (0.2), floored at `min_n_mcmc=4` and
capped at `max_n_mcmc=16`. This is what CLAUDE.md prescribes, and `adaptive_rejuvenation.ipynb` is the reason
the cap is set high rather than a small fixed `n_mcmc` being used: there, at $\lambda = 1.25$, 27–36 sweeps
still recovered only **5–6%** of the grid $E_{q}[f_{2}]$ (3.48 vs 66.11). Too few sweeps is a *mixing*
failure, and it under-tilts — no amount of reward fidelity compensates for it.

pCN's `decorrelation` is a **normalized ESJD in latent $z$**, built to be dimension-robust (a raw
$\lvert\cos\rvert$ would floor at a $d$-dependent value), so the 2D-calibrated 0.2 threshold is *meant* to
carry over to $d = 65536$. Whether it does is an open question, and `stop_reason_history` answers it:
`"decorr"` means the rule fired on its own, `"cap"` means 16 sweeps were not enough. (On the 2D ring toy at
$\lambda = 6$ with this same cap, 7 of 8 levels hit `"cap"`.) Note also that the `plateau` early-exit visible
in `adaptive_rejuvenation.ipynb`'s recorded output **no longer exists** (removed in `8549e1a`), so expect
higher sweep counts here than that notebook shows.

### Surviving a run that does not finish

The number of levels is not knowable in advance, and `killable` is preemptible — so this run may well be cut
short. `CheckpointingPCNKernel` below subclasses `PCNKernel` to persist the particle cloud **after every
sweep**, so nothing is lost:

- It keys snapshots by **β**, which changes only between levels. Each entry is overwritten within a level and
  therefore ends up holding that level's *last* sweep — exactly the post-rejuvenation cloud the final
  snapshot records. (Verified on the 2D toy: the partial file's `X` equals `levels[k].X` at every level.)
- It only *reads* state after `super().step()` has run, so no randomness is consumed and no value changes.
  The run is **bit-identical** to one using a plain `PCNKernel` — verified against `X`, `logw`, `betas`,
  `ess_history`, `n_mcmc_history` and `stop_reason_history`.
- It writes to a temp file and `os.replace`s it, so a kill *during* a write cannot corrupt the last good file.
- The partial file uses the **same schema** as the final checkpoint, so `decode_levels.py` reads it unchanged:
  `cd notebooks/flux_lambda_sweep && python decode_levels.py partial_N8_seed101.pt --device cpu`
- With `SAVE_LATENT_Z=True` it also stores the pCN latent `z`, so a killed run is *resumable* in principle,
  not merely decodable.

Two things only the **final** checkpoint has: the true `ancestors` (resampling happens outside the kernel, so
the partial file fills them with `arange` and lineage column ordering degenerates to raw order), and the
β=0 anchor level (`step` is never called at β=0).

In [ ]:
from creativity_measure import LevelSnapshot, PCNKernel, adaptive_tempering_smc_sample

# N, MAX_SWEEPS and the IEM knobs are set in the cost-knobs cell; DEADLINE in the environment cell.
ESS_TARGET = 0.85
SEED = 101
SAVE_LATENT_Z = True    # also persist the pCN latent z -> a stopped run becomes resumable, not just
                        # decodable. Roughly doubles the partial file (still tens of MB).

CONFIG = {
    "model_id": MODEL_ID,            # which VAE to decode these latents with
    "img": IMG, "C": C, "H": H, "W": W, "latent_dim": LATENT_DIM,
    "vae_scaling_factor": VAE_SF, "vae_shift_factor": VAE_SHIFT,
    "N": N, "lam_max": LAM_MAX, "lam_s": lam_s, "ess_target": ESS_TARGET,
    "n_mcmc": None, "max_n_mcmc": MAX_SWEEPS, "seed": SEED,
    "prompt": PROMPT, "guidance": GUIDANCE, "n_steps": N_STEPS,
    "sigma_min": SIGMA_MIN, "sigma_max": SIGMA_MAX,
    "R": R, "n_gamma": N_GAMMA, "num_eps": NUM_EPS,
    "t_norm_s": t_norm, "t_sweep_s": t_sweep, "t_per_pass_s": t_per_pass,
}

PARTIAL_CKPT = os.path.join(SWEEP_DIR, f"partial_N{N}_seed{SEED}.pt")


class DeadlineReached(RuntimeError):
    """Raised by the kernel when the wall-clock deadline passes, to stop the SMC cleanly."""


class CheckpointingPCNKernel(PCNKernel):
    """pCN kernel that persists the cloud every sweep and stops the run at a wall-clock deadline.

    Why: snapshots are otherwise only returned when `adaptive_tempering_smc_sample` *finishes*, and
    the number of levels is not knowable in advance -- so a timeout would otherwise lose everything.

    How it stays honest: it only *reads* state after `super().step()` has done its work -- no RNG is
    consumed, no value is altered -- so the run is bit-identical to one using a plain PCNKernel.

    Keying by `beta` is the trick: beta changes only between tempering levels, so each entry is
    repeatedly overwritten within a level and ends up holding that level's LAST sweep, i.e. exactly
    the post-rejuvenation cloud the final snapshots record. Dicts keep insertion order and beta
    increases, so the saved list is already in ascending lambda_eff order. `init` is overridden too,
    so the beta=0 / lambda_eff=0 anchor level is captured as well.

    The file uses the same schema as the final checkpoint, so `decode_levels.py` reads a stopped run
    unchanged. One caveat: `ancestors` is filled with arange (resampling happens outside the kernel,
    so true lineage is unavailable here -> lineage column ordering degenerates to raw order).
    """

    def __init__(self, *args, path: str, config: dict, deadline: float,
                 save_z: bool = True, **kwargs):
        super().__init__(*args, **kwargs)
        self._path = path
        self._config = config
        self._deadline = deadline
        self._save_z = save_z
        self._latest: dict[float, dict] = {}
        self._sweeps_at: dict[float, int] = {}
        self._sweeps = 0

    def _record(self, state, *, beta: float, lam: float, acc: float) -> None:
        entry = {
            "beta": beta,
            "lam_eff": beta * lam,
            "X": state.X.detach().to("cpu").clone(),        # data space -> VAE-decodable
            "fX": state.fX.detach().to("cpu").clone(),
            "logw": state.logw.detach().to("cpu").clone(),
            "ancestors": torch.arange(state.X.shape[0]),
            "sweeps_at_beta": self._sweeps_at.get(beta, 0),
            "acc": acc,
        }
        if self._save_z:
            entry["aux"] = {k: v.detach().to("cpu").clone() for k, v in state.aux.items()}
        self._latest[beta] = entry
        # Write-then-rename: a kill DURING the write cannot corrupt the last good file.
        tmp = f"{self._path}.tmp"
        torch.save(
            {
                "levels": list(self._latest.values()),
                "betas": list(self._latest.keys()),
                "config": self._config,
                "partial": True,
                "sweeps_done": self._sweeps,
            },
            tmp,
        )
        os.replace(tmp, self._path)

    def init(self, n_particles, f, *, generator, device, dtype):
        state = super().init(n_particles, f, generator=generator, device=device, dtype=dtype)
        self._record(state, beta=0.0, lam=0.0, acc=float("nan"))   # the free lambda_eff=0 anchor
        return state

    def step(self, state, *, beta, lam, f, generator):
        acc = super().step(state, beta=beta, lam=lam, f=f, generator=generator)
        b = float(beta)
        self._sweeps += 1
        self._sweeps_at[b] = self._sweeps_at.get(b, 0) + 1
        self._record(state, beta=b, lam=float(lam), acc=float(acc))
        if time.time() > self._deadline:
            raise DeadlineReached(
                f"wall-clock deadline reached after {self._sweeps} sweeps "
                f"at beta={b:.4f}; {len(self._latest)} levels are on disk"
            )
        return acc


kernel = CheckpointingPCNKernel(
    G, LATENT_DIM, path=PARTIAL_CKPT, config=CONFIG, deadline=DEADLINE, save_z=SAVE_LATENT_Z
)

print(f"SMC: N={N} lam={LAM_MAX:.1f} ess_target={ESS_TARGET} "
      f"n_mcmc=None (adaptive, cap {MAX_SWEEPS})")
print(f"checkpoint every sweep -> {PARTIAL_CKPT}")
print(f"will stop cleanly at {time.strftime('%H:%M:%S', time.localtime(DEADLINE))} "
      f"({(DEADLINE - time.time()) / 3600:.2f} h from now)")

t0 = time.time()
timed_out = False
try:
    res = adaptive_tempering_smc_sample(
        reward,
        lam=LAM_MAX,
        n_particles=N,
        kernel=kernel,
        ess_target=ESS_TARGET,
        n_mcmc=None,               # adaptive -- let the decorrelation rule decide
        max_n_mcmc=MAX_SWEEPS,
        keep_levels=True,
        snapshot_device="cpu",     # keep 8 x 65536 fp32 per level off the GPU
        seed=SEED,
    )
    levels = res.levels
    assert levels is not None
except DeadlineReached as exc:
    timed_out = True
    print(f"\n*** STOPPED ON DEADLINE: {exc}")
    print("*** Rebuilding levels from the partial checkpoint; the cells below will render them.")
    _entries = list(kernel._latest.values())
    levels = [
        LevelSnapshot(beta=e["beta"], lam_eff=e["lam_eff"], X=e["X"], fX=e["fX"],
                      logw=e["logw"], ancestors=e["ancestors"])
        for e in _entries
    ]

    class _PartialResult:
        """Stands in for AdaptiveTemperingSMCResult so the decode/diagnostics cells work unchanged.

        ESS is computed inside the SMC loop, not the kernel, so it is unavailable here (NaN).
        Stop reasons are INFERRED: a level that used all MAX_SWEEPS sweeps is 'cap', a shorter one
        stopped on 'decorr', and the level in progress when the deadline hit is 'timeout'.
        """

        def __init__(self, entries):
            tail = entries[1:]                      # entries[0] is the beta=0 anchor
            self.betas = [e["beta"] for e in tail]
            self.ess_history = [float("nan")] * len(tail)
            self.acc_history = [e["acc"] for e in tail]
            self.n_mcmc_history = [e["sweeps_at_beta"] for e in tail]
            self.stop_reason_history = [
                ("cap" if e["sweeps_at_beta"] >= MAX_SWEEPS else "decorr") for e in tail
            ]
            if self.stop_reason_history:
                self.stop_reason_history[-1] = "timeout"
            self.X = levels[-1].X
            self.logw = levels[-1].logw
            self.levels = levels

    res = _PartialResult(_entries)

mins = (time.time() - t0) / 60
print(f"\n{'TIMED OUT' if timed_out else 'completed'} after {mins:.1f} min ({mins / 60:.2f} h)")
print(f"levels={len(levels)}  betas={len(res.betas)}")
print(f"total sweeps  = {sum(res.n_mcmc_history)}")
print(f"per level     = {res.n_mcmc_history}")
print(f"stop reasons  = {res.stop_reason_history}"
      + ("   (inferred -- see _PartialResult)" if timed_out else ""))
print(f"measured cost = {mins * 60 / max(sum(res.n_mcmc_history), 1):.0f} s/sweep "
      f"(preflight predicted {SWEEP_PASSES * t_per_pass:.0f} s)")

# Does the 2D-calibrated decorrelation threshold fire at d=65536?
# (On the 2D ring toy with lam=6 and this same cap, 7 of 8 levels hit 'cap'.)
n_capped = sum(1 for r in res.stop_reason_history if r == "cap")
n_total = len(res.stop_reason_history)
print(f"levels hitting the cap: {n_capped}/{n_total}")
if n_total and n_capped == n_total:
    print("  -> 'decorr' NEVER fired: 16 sweeps do not decorrelate at this dimension.")
elif n_capped:
    print("  -> mixed: the hard (high-beta) levels need the cap, easier ones decorrelate on their own.")
else:
    print("  -> 'decorr' fired throughout: the 2D-calibrated threshold transfers to d=65536.")

# Final checkpoint. On a completed run this is authoritative and carries true `ancestors`; on a
# timeout it mirrors the partial file (arange ancestors) and is flagged as such.
suffix = "_TIMEOUT" if timed_out else ""
ckpt = os.path.join(SWEEP_DIR, f"levels_N{N}_lam{LAM_MAX:.0f}_seed{SEED}{suffix}.pt")
torch.save(
    {
        "levels": [
            {"beta": s.beta, "lam_eff": s.lam_eff, "X": s.X, "fX": s.fX,
             "logw": s.logw, "ancestors": s.ancestors}
            for s in levels
        ],
        "X": res.X,
        "logw": res.logw,
        "betas": res.betas,
        "ess_history": res.ess_history,
        "acc_history": res.acc_history,
        "n_mcmc_history": res.n_mcmc_history,
        "stop_reason_history": res.stop_reason_history,
        "partial": timed_out,
        "config": {**CONFIG, "wall_clock_min": mins, "timed_out": timed_out},
    },
    ckpt,
)
print(f"\nsaved -> {ckpt}  ({os.path.getsize(ckpt) / 2**20:.1f} MB)")

print(f"\n{'k':>3} {'beta':>8} {'lam_eff':>10} {'E_q[f]':>9} {'sweeps':>7} {'stop':>8}")
for k, s in enumerate(levels):
    sw = res.n_mcmc_history[k - 1] if k > 0 else 0
    st = res.stop_reason_history[k - 1] if k > 0 else "-"
    print(f"{k:3d} {s.beta:8.4f} {s.lam_eff:10.1f} {float(s.fX.mean()):9.4f} {sw:7d} {st:>8}")

## 6. Decode and render the $\lambda_{\text{eff}} \times$ particle grid

Rows are $\lambda_{\text{eff}}$ ascending (top row is the free $\lambda = 0$ anchor, i.e. plain samples from
$p$). Columns are ordered **by lineage**: starting from the final level we walk `ancestors` backwards, so
column $j$ tracks one particle's drift as the tilt strengthens.

Where two columns show the same image, that is **coalescence** — the resampler killed one lineage and
duplicated another. Making it visible is the point: it is the honest picture of how much independent
information the ensemble actually carries at strong tilts.

In [ ]:
# --- lineage columns: walk `ancestors` backwards from the final level ---------------------------
# col_of[k][j] = index, within levels[k].X, of the ancestor of final-level particle j.
col_of = [None] * len(levels)
cur = torch.arange(N)
col_of[-1] = cur
for k in range(len(levels) - 1, 0, -1):
    cur = levels[k].ancestors[cur]      # parents of `cur` in levels[k-1]
    col_of[k - 1] = cur

# How many of the final N lineages are still distinct at each level. This is a column-collapse
# statistic for READING THE GRID -- it is not the uniq/N diversity diagnostic (see the next
# section): col_of[-1] is arange(N) by construction, so this necessarily reads N at the last level.
lineage_uniq = [int(torch.unique(c).numel()) for c in col_of]
print("distinct final-level lineages present at each level:", lineage_uniq)

# --- decode every level ------------------------------------------------------------------------
print(f"\nDecoding {len(levels)} levels x {N} particles = {len(levels) * N} images ...")
t0 = time.time()
decoded = []
for k, s in enumerate(levels):
    decoded.append(decode(s.X[col_of[k]]))     # already in lineage column order
    print(f"  level {k:2d}  lam_eff={s.lam_eff:8.1f}  ok")
print(f"decoded in {(time.time() - t0) / 60:.1f} min")

# --- the grid ----------------------------------------------------------------------------------
nrow, ncol = len(levels), N
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.9, nrow * 2.05))
axes = axes.reshape(nrow, ncol)
for k in range(nrow):
    for j in range(ncol):
        ax = axes[k, j]
        ax.axis("off")
        ax.imshow(decoded[k][j].permute(1, 2, 0).clamp(0, 1).numpy())
        if j == 0:
            ax.text(
                -0.12, 0.5,
                f"$\\lambda_{{eff}}$={levels[k].lam_eff:.0f}\n$\\beta$={levels[k].beta:.3f}",
                transform=ax.transAxes, ha="right", va="center", fontsize=8,
            )
        if k == 0:
            ax.set_title(f"col {j}", fontsize=8)
fig.suptitle(
    f"One SMC run at $\\lambda_{{max}}$={LAM_MAX:.0f} -> a free $\\lambda$ sweep  "
    f"(prompt: {PROMPT!r}, N={N})\nrows: $\\lambda_{{eff}}$ ascending   "
    f"columns: lineage (same column = one particle's ancestry)",
    fontsize=11,
)
plt.tight_layout(rect=(0.03, 0, 1, 0.97))
plt.show()

## 7. Diagnostics and λ selection

All of this comes from data already in the result — **no extra reward calls**.

The established selection rule (2D-validated, recorded in CLAUDE.md): $E_{q}[f]$ rises forever with λ, so
novelty is *not* the stopping signal — **degeneracy is**. Pick the largest λ whose ensemble diversity
(`uniq/N`, min per-level ESS/N) stays above ~0.5.

`uniq/N` follows the repo convention — distinct particle rows **within** each level (`X.unique(dim=0)`) —
deliberately *not* the lineage walk from the previous cell, which reads 1.00 at the final level by
construction and would make the selector always choose the last row.

`uniq/N` still has a blind spot: pCN rejuvenation nudges duplicated particles apart, so a collapsed ensemble
can present as N distinct rows. **f-spread** ($\operatorname{std}_{q}(f)$) and ESS/N are the corroborating
signals, which is why all three are plotted.

In [ ]:
lam_effs = [s.lam_eff for s in levels]
Eqf = [float(s.fX.mean()) for s in levels]

# Repo convention (tiny-SD notebook, cell 15): uniq/N counts distinct particle ROWS WITHIN a level,
# and f-spread is the within-ensemble reward std.
#
# Deliberately NOT the lineage walk: col_of[-1] is arange(N) by construction, so a lineage-based
# uniq/N reads 1.00 at the final level no matter how degenerate it is -- which would make the
# selector below always pick the last level. The lineage walk is for ordering grid columns only.
uniq_frac = [int(s.X.unique(dim=0).shape[0]) / s.X.shape[0] for s in levels]
f_spread = [float(s.fX.std()) for s in levels]

# res.ess_history / acc_history are per *transition* (len == len(betas)); level 0 has none.
# NOTE: ess_history holds RAW ESS counts in [1, N] (_ess_from_logw), so normalize by N to compare
# against the 0.5 diversity floor. `ess_target` is separately a *fraction* -- don't confuse them.
ess_hist = [float("nan")] + [float(e) / N for e in res.ess_history]
acc_hist = [float("nan")] + [float(a) for a in res.acc_history]
assert len(ess_hist) == len(acc_hist) == len(levels)
assert all(math.isnan(e) or 0.0 <= e <= 1.0 + 1e-9 for e in ess_hist), "ESS/N out of range"

base_f = Eqf[0]
print(f"base (lambda_eff=0) mean f = {base_f:.4f}   ess_target = {ESS_TARGET} (fraction of N)\n")
print(f"{'k':>3} {'lam_eff':>9} {'m':>6} {'beta':>7} {'E_q[f]':>8} {'d f':>8} "
      f"{'ESS/N':>7} {'acc':>6} {'uniq/N':>7} {'f-spread':>9}")
for k in range(len(levels)):
    print(f"{k:3d} {lam_effs[k]:9.1f} {lam_effs[k] / lam_s:6.2f} {levels[k].beta:7.4f} "
          f"{Eqf[k]:8.4f} {Eqf[k] - base_f:+8.4f} {ess_hist[k]:7.2f} {acc_hist[k]:6.2f} "
          f"{uniq_frac[k]:7.2f} {f_spread[k]:9.4f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].plot(lam_effs, Eqf, "o-")
ax[0].set_xlabel(r"$\lambda_{eff}$"); ax[0].set_ylabel(r"$E_q[f]$")
ax[0].set_title("novelty rises with the tilt (never a stop signal)")

ax[1].plot(lam_effs, uniq_frac, "o-", label="uniq/N")
ax[1].plot(lam_effs, ess_hist, "s--", label="ESS/N")
ax[1].axhline(0.5, color="crimson", ls=":", label="0.5 floor")
ax[1].set_xlabel(r"$\lambda_{eff}$"); ax[1].set_ylim(0, 1.05)
ax[1].legend(fontsize=8); ax[1].set_title("diversity IS the stop signal")

ax[2].plot(lam_effs, f_spread, "o-", color="darkgreen")
ax[2].set_xlabel(r"$\lambda_{eff}$"); ax[2].set_ylabel(r"std$_q(f)$")
ax[2].set_title("f-spread: collapse that uniq/N can miss")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# --- apply the established rule ----------------------------------------------------------------
ok = [k for k in range(len(levels))
      if uniq_frac[k] >= 0.5 and (math.isnan(ess_hist[k]) or ess_hist[k] >= 0.5)]
if ok:
    best = max(ok)
    print(f"\nSelected: level {best},  lambda_eff = {lam_effs[best]:.1f} "
          f"(m = {lam_effs[best] / lam_s:.2f})")
    print(f"  uniq/N={uniq_frac[best]:.2f}  ESS/N={ess_hist[best]:.2f}  "
          f"E_q[f]={Eqf[best]:.4f} ({Eqf[best] - base_f:+.4f} vs base)  "
          f"f-spread={f_spread[best]:.4f}")
    print(f"  Largest lambda_eff whose diversity stays above 0.5; "
          f"{len(levels) - 1 - best} level(s) above it are degenerate.")
else:
    print("\nNo level clears the 0.5 diversity floor -- the tilt is too strong for N=8 everywhere.")
    print("Either lower LAM_MAX or raise N.")

# uniq/N has a blind spot worth stating explicitly in the Takeaways.
if any(u == 1.0 and s < 1e-3 * max(f_spread) for u, s in zip(uniq_frac, f_spread)):
    print("\nNOTE: a level shows uniq/N=1.00 with near-zero f-spread. pCN rejuvenation moves duplicated")
    print("      particles apart, so distinct rows can hide a collapsed ensemble. Trust f-spread/ESS.")

## Takeaways

*(Fill in after the run — this cell is the authoritative statement of what was learned, per repo convention.)*

**What the mechanism buys.** One run at $\lambda_{\max}$ replaced N separate runs. Every row shares the same
reward, references, base density and Brownian seed, so differences between rows are attributable to
$\lambda_{\text{eff}}$ alone — which separate runs, each with its own β schedule and ancestry, can never quite
claim. The $\lambda = 0$ anchor row came for free.

**What it does not buy — read this before quoting any row as a sample from $q_{\lambda}$:**

- **Rows are not independent draws.** Level $k$ and $k+1$ differ by a few pCN sweeps from a shared ancestry;
  consecutive rows are strongly correlated. The lineage column ordering exposes this rather than hiding it.
- **λ_eff values are not ours to choose.** The ESS rule picks them, so the grid is unevenly spaced in λ and a
  specific λ of interest will generally not appear. Forced β checkpoints were explicitly out of scope.
- **`final_resample=True` makes `res.X` differ from `levels[-1].X`** — the former is resampled, the latter is
  not. Documented, not reconciled.

**To record after running:**

- σ↔t mapping validated? (go/no-go images clean and on-prompt — cell 7)
- Levels obtained, and λ_eff spacing at `ess_target=0.85`
- Selected λ by the diversity ≥ 0.5 rule, and where degeneracy set in
- Wall-clock, and measured cost per level vs the ≈2.4k transformer-pass budget estimate